In [1]:
!git clone --depth 1 -b main https://github.com/FatiAllali123/SLM-Software-Engineering.git /kaggle/working/SLM-Software-Engineering
!cp /kaggle/working/SLM-Software-Engineering/finetuning/prepare_data.py /kaggle/working/prepare_data.py
!python /kaggle/working/prepare_data.py 2>&1 | grep -E "removed total|train token stats|wrote"

import hashlib
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
EXPECTED = {"train.jsonl": "8b2d6e743b8f8a23b89d20f1cc0f97d823a522134fe291e5aab0655dc7df2b2a",
            "val.jsonl":   "01b59b3f8b4ad639993e20f566038d3942c3c55f291e5f4d4fbbff404e3cc116"}
for f, h in EXPECTED.items():
    assert sha(f"/kaggle/working/ft_data/{f}") == h, f"{f} differs from the training data"
print("data fingerprint OK")

Cloning into '/kaggle/working/SLM-Software-Engineering'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 85 (delta 26), reused 39 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 214.83 KiB | 2.68 MiB/s, done.
Resolving deltas: 100% (26/26), done.
removed total 3562 (n-gram 3326, name 361, both 125) -> kept 34722
train token stats: {'mean': 523.5, 'median': 508, 'max': 1698} | too long dropped: 1 of 6301
wrote /kaggle/working/ft_data
data fingerprint OK


In [2]:
import json, re, math, os
from huggingface_hub import HfApi

# ---- 1. Pin the sources (record what exists today) ----
api = HfApi(); rev = {}
for kind, repo in [("dataset", "ise-uiuc/Magicoder-OSS-Instruct-75K"),
                   ("dataset", "openai/openai_humaneval"),
                   ("model",   "Qwen/Qwen2.5-Coder-1.5B-Instruct")]:
    info = api.dataset_info(repo) if kind == "dataset" else api.model_info(repo)
    rev[repo] = {"kind": kind, "sha": info.sha, "last_modified": str(info.last_modified)}
print(json.dumps(rev, indent=1))

# ---- 2. Secret / PII scan over train + val (problem and solution text) ----
PATTERNS = {
  "aws_access_key":   r"\bAKIA[0-9A-Z]{16}\b",
  "github_token":     r"\bgh[pousr]_[A-Za-z0-9]{36,}\b",
  "slack_token":      r"\bxox[baprs]-[A-Za-z0-9-]{10,}\b",
  "google_api_key":   r"\bAIza[0-9A-Za-z_\-]{35}\b",
  "stripe_live_key":  r"\bsk_live_[0-9a-zA-Z]{24,}\b",
  "openai_style_key": r"\bsk-[A-Za-z0-9]{32,}\b",
  "private_key_block": r"-----BEGIN (?:RSA |EC |DSA |OPENSSH )?PRIVATE KEY-----",
  "jwt":              r"\beyJ[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}\b",
  "secret_assignment": r"(?i)\b(?:api[_-]?key|secret|token|passwd|password|access[_-]?key)\w*\s*[=:]\s*['\"]([^'\"\s]{12,})['\"]",
  "email":            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
}
PLACEHOLDER = re.compile(r"(?i)example|your|xxx|changeme|dummy|placeholder|test|sample|fake|<|\{|\$|1234|abcd|secret|password|token")

def entropy(s):
    return -sum((s.count(c)/len(s)) * math.log2(s.count(c)/len(s)) for c in set(s))

counts = {k: 0 for k in PATTERNS}; flagged = {}   # row_id -> reasons
n_rows = 0
for fname in ("train.jsonl", "val.jsonl"):
    for line in open(f"/kaggle/working/ft_data/{fname}", encoding="utf-8"):
        ex = json.loads(line); n_rows += 1
        text = "\n".join(m["content"] for m in ex["messages"])
        for name, pat in PATTERNS.items():
            for m in re.finditer(pat, text):
                val = m.group(1) if name == "secret_assignment" else m.group(0)
                counts[name] += 1
                if name == "email":
                    continue                       # counted only, reported as PII
                looks_real = (name != "secret_assignment") or (
                    not PLACEHOLDER.search(val) and len(val) >= 20 and entropy(val) >= 3.5)
                if looks_real:
                    flagged.setdefault(ex["row_id"], []).append(name)

print("rows scanned:", n_rows)
print("hits per pattern:", counts)
print("examples with credential-like candidates (not placeholders):", len(flagged))
for rid, why in list(flagged.items())[:15]:
    print(" row_id", rid, "->", sorted(set(why)))

os.makedirs("/kaggle/working/out", exist_ok=True)
json.dump(rev, open("/kaggle/working/out/source_revisions.json", "w"), indent=2)
json.dump({"rows_scanned": n_rows, "hits_per_pattern": counts,
           "n_flagged_examples": len(flagged), "flagged_row_ids": sorted(flagged),
           "note": "regex + entropy heuristic; no secret values stored"},
          open("/kaggle/working/out/secret_scan_report.json", "w"), indent=2)
print("saved to /kaggle/working/out")

{
 "ise-uiuc/Magicoder-OSS-Instruct-75K": {
  "kind": "dataset",
  "sha": "5f839b1f368a76b161028bb9edff055db34022b2",
  "last_modified": "2023-12-04 10:35:04+00:00"
 },
 "openai/openai_humaneval": {
  "kind": "dataset",
  "sha": "7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544",
  "last_modified": "2024-01-04 16:08:05+00:00"
 },
 "Qwen/Qwen2.5-Coder-1.5B-Instruct": {
  "kind": "model",
  "sha": "2e1fd397ee46e1388853d2af2c993145b0f1098a",
  "last_modified": "2025-01-12 02:05:01+00:00"
 }
}
rows scanned: 6300
hits per pattern: {'aws_access_key': 0, 'github_token': 0, 'slack_token': 0, 'google_api_key': 0, 'stripe_live_key': 0, 'openai_style_key': 0, 'private_key_block': 0, 'jwt': 1, 'secret_assignment': 20, 'email': 157}
examples with credential-like candidates (not placeholders): 1
 row_id 47234 -> ['jwt']
saved to /kaggle/working/out


In [3]:
import json, re, base64, collections
def b64j(s): return json.loads(base64.urlsafe_b64decode(s + "=" * (-len(s) % 4)))
JWT = re.compile(r"\beyJ[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}\b")
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@([A-Za-z0-9.-]+\.[A-Za-z]{2,})\b")
doms = collections.Counter()
for fname in ("train.jsonl", "val.jsonl"):
    for line in open(f"/kaggle/working/ft_data/{fname}", encoding="utf-8"):
        ex = json.loads(line); text = "\n".join(m["content"] for m in ex["messages"])
        for m in EMAIL.finditer(text): doms[m.group(1).lower()] += 1
        if ex["row_id"] == 47234:
            for m in JWT.finditer(text):
                h, p, sig = m.group(0).split(".")
                try:
                    print("header:", b64j(h))
                    pl = b64j(p); print("payload (values cut):", {k: str(v)[:20] for k, v in pl.items()})
                except Exception as e: print("not decodable JSON:", type(e).__name__)
                print("signature length:", len(sig))
                i = m.start(); print("context before:", repr(text[max(0, i-120):i]))
print("email domains:", doms.most_common(15), "| distinct:", len(doms))

header: {'alg': 'HS256', 'typ': 'JWT'}
payload (values cut): {'sub': '1234567890', 'name': 'John Doe', 'iat': '1516239022'}
signature length: 43
context before: 'ndle potential exceptions. You should use this as a reference to implement the function.\n\nExample:\nInput:\nauth_token = "'
email domains: [('example.com', 133), ('github.com', 7), ('gmail.com', 7), ('email.com', 5), ('yahoo.com', 3), ('outlook.com', 1), ('company.co.uk', 1)] | distinct: 7
